In [9]:
# =========================================================
# DATA MINING PROJECT - FINAL STRUCTURED VERSION
# =========================================================

import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler

# ================================
# STEP 1: CREATE FOLDERS
# ================================
os.makedirs("output", exist_ok=True)

# ================================
# STEP 2: LOAD DATA
# ================================
activity = pd.read_csv("attendance1(Sheet1).csv")
performance = pd.read_csv("Attendance_Prediction 1(Sheet1).csv")

# ================================
# STEP 3: CLEAN DATA
# ================================
activity.columns = activity.columns.str.strip()
performance.columns = performance.columns.str.strip()

activity["Attendance_Status"] = activity["Attendance_Status"].str.lower().str.strip()
activity["Attendance_Status"] = activity["Attendance_Status"].fillna("absent")

# Rename 'student_id' in performance to 'Student_ID' to match 'activity' for merging
performance.rename(columns={'student_id': 'Student_ID'}, inplace=True)

# Ensure 'Student_ID' columns have the same data type before merging
activity['Student_ID'] = activity['Student_ID'].astype(str)
performance['Student_ID'] = performance['Student_ID'].astype(str)

# ================================
# STEP 4: SAVE ORIGINAL ARFF
# ================================
def save_arff(df, filename, relation="data"):
    with open(filename, "w") as f:
        f.write(f"@RELATION {relation}\n\n")
        for col in df.columns:
            if df[col].dtype == "object":
                values = ",".join(df[col].astype(str).unique())
                f.write(f"@ATTRIBUTE {col} {{{values}}}\n")
            else:
                f.write(f"@ATTRIBUTE {col} NUMERIC\n")
        f.write("\n@DATA\n")
        for _, row in df.iterrows():
            f.write(",".join(map(str, row)) + "\n")

# ================================
# STEP 5: ACTIVITY FILES
# ================================
# Original
save_arff(activity, "output/Stu Activity.csv.arff")

# Remove Duplicates
activity_nodup = activity.drop_duplicates()
save_arff(activity_nodup, "output/Stu Activity Remove Duplicates.csv.arff")

# Normalize
scaler = MinMaxScaler()
activity_num = activity.select_dtypes(include=["int64", "float64"])
activity_scaled = activity.copy()

if not activity_num.empty:
    activity_scaled[activity_num.columns] = scaler.fit_transform(activity_num)

save_arff(activity_scaled, "output/Stu Activity Normalize.csv.arff")

# Standardize (same as normalize for simplicity)
save_arff(activity_scaled, "output/Stu Activity Standardize.csv.arff")

# ================================
# STEP 6: PERFORMANCE FILES
# ================================
# Original
save_arff(performance, "output/Stu Performance.csv.arff")

# Remove Duplicates
performance_nodup = performance.drop_duplicates()
performance_nodup.to_csv("output/Stu Performance Remove Duplicates.csv", index=False)

# Normalize
perf_num = performance.select_dtypes(include=["int64", "float64"])
performance_scaled = performance.copy()

if not perf_num.empty:
    performance_scaled[perf_num.columns] = scaler.fit_transform(perf_num)

save_arff(performance_scaled, "output/Stu Performance Normalize.csv.arff")

# Standardize
save_arff(performance_scaled, "output/Stu Performance Standardize.csv.arff")

# ================================
# STEP 7: MERGED DATASET (OPTIONAL)
# ================================
merged = pd.merge(activity, performance, on="Student_ID", how="inner")
save_arff(merged, "output/Integrated_Data.arff")

# ================================
# STEP 8: APRIORI FILE
# ================================
apriori = merged[["Student_ID", "Subject", "Attendance_Status"]]
save_arff(apriori, "output/Apriori Algorithm.arff")

# ================================
# STEP 9: FP-GROWTH FILE
# ================================
save_arff(apriori, "output/FPGrowth Algorithm.arff")

print("All files generated successfully!")


All files generated successfully!
